# Per-pore analysis from XCT volumes (Defect Atlas backend)

Receives a **list of folders**; in each one it finds the `*_onlypores.tif` and matching `*_samplemask.tif` volumes and computes, for **every pore** in the specimen, geometric / shape / orientation / distance-to-surface / border / classification metrics, plus specimen-level summaries (void volume fraction, counts by type, key figures, histograms).

**Nothing is reimplemented.** The notebook imports the existing Defect Atlas backend (`app.…`, read-only) and calls:

| Step | Backend function |
|---|---|
| labelling + all properties + border flags | `PipelineExecutor.run_pipeline` |
| Pore / LoF / Cluster / Border | `app.domain.services.classification.classify_defects` |
| shape class (Spherical / Disc / …) | `app.domain.services.shape_classifier.classify_shape` (inputs converted with `app.routers.study._covariance_ratio_to_axis_ratio`) |
| key figures | `app.services.analytics_service._volume_key_figures` |
| per-slice porosity | `app.services.porosity_report_service.compute_direction_report` |
| Z projections | `app.domain.services.projections.compute_projections` |

**Inputs are TIFF** (`*_onlypores.tif`, `*_samplemask.tif`, as written by `onlypores.ipynb`). The backend only reads RAW, so they are streamed slice-by-slice into polarity-normalised RAW files in `OUT_DIR/_raw/` (pores = 255, material = 0). Axis convention is the TIFF's: `(Z, Y, X)` = `(page, row, column)`.

**Kernel:** `defect-atlas` (Python 3.11, pydantic 1.x, numpy 1.26). The default `prep` kernel cannot import the backend.

## 1. Parameters

In [1]:
# ----------------------------- INPUTS ---------------------------------------
# Folders to analyse. Each is searched (recursively) for '*_onlypores.tif' and the matching '*_samplemask.tif'
# (same prefix, as written by onlypores.ipynb). Every pair found is analysed; results go to <pores folder>/pore_analysis.
FOLDERS = [
    "/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/4/onlypores files",
    "/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/5/onlypores files",
    "/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/7/onlypores files",
    "/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/8/onlypores files",
    "/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/10/onlypores files",
    "/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/11/onlypores files",
    "/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/12/onlypores files",
]
PORES_GLOB = "*_onlypores.tif"
MASK_SUFFIX = "_samplemask.tif"    # mask name = pores name with '_onlypores.tif' replaced by this
OUT_SUBDIR = "pore_analysis"       # output folder name, created next to each pores TIFF
STOP_ON_ERROR = False              # False = log the error and continue with the next folder

VOXEL_SIZE_MM = 0.025          # default isotropic voxel size in mm (0.025 mm = 25 µm)
VOXEL_SIZE_OVERRIDES = {}      # per-folder/prefix override, e.g. {"Na_01_1": 0.02}; key = substring of the pores TIFF path
GRAY_VOLUME_PATH  = None       # not supported in batch mode (kept for run_config compatibility)
INVERT_PORES = None            # None = auto-detect polarity; True/False = force
INVERT_MASK  = None            # (polarity = which value means "pore"/"material")
WALLS = "auto"                 # front/back wall exclusion along the TIFF Z axis (pages):
                               #   "auto" = read 'Front wall slice' / 'Back wall slice' from the *_report.txt next to the pores TIFF
                               #   (front, back) = explicit slice indices for ALL volumes;  None = no wall exclusion
CROP = None                    # None, or ((z0, z1), (y0, y1), (x0, x1)) in voxels, for quick tests

# ----------------------------- PIPELINE -------------------------------------
MIN_SIZE_VOXELS = 0            # kill_small   (0 = keep everything)
MAX_SIZE_VOXELS = None         # kill_big     (None = no limit)
CONNECTIVITY    = 26
ENABLE_MERGING  = False        # merge nearby defects into clusters (recomputes properties)
ENABLE_LOF      = True         # False => no pore can become LoF (only Pore/Cluster/Border)

# classification (classify_defects defaults)
LOF_VOLUME_THRESHOLD_VOX   = 60.0
LOF_ELONGATION_THRESHOLD   = 0.7
LOF_DISTANCE_THRESHOLD_VOX = 100.0
CPX_THRESHOLD = 0.7001         # classify_shape default: cpx_factor above this => "Complex"

# ----------------------------- OUTPUT / RUN ---------------------------------
USE_CACHE = True               # reuse raw conversion + pipeline result if inputs/params unchanged
KEEP_RAW = True                # keep OUT/_raw/*.raw (needed for cache reuse, ~1 byte/voxel each)
RUN_SLICE_PROFILE = True
SLICE_AXES = ("z", "y", "x")   # per-slice porosity directions
RUN_PROJECTION = True
SHOW_PLOTS = True              # False = save figures only (faster / cleaner for many folders)
BATCH_SUMMARY_PATH = None      # None => <first folder>/pore_analysis_batch_summary.csv

# ----------------------------- SETUP PATHS ----------------------------------
REPO = "/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/10_code/Defect Atlas"


## 2. Environment and import setup

* Sets `DEFECT_ATLAS_ACCELERATION_MODE=cpu` *before* importing (no `cupy` in this env, so the CUDA path would only fail and fall back).
* Sets `sys.dont_write_bytecode` so **no `__pycache__` is written inside `REPO`** (read-only use).
* Checks that the whole `REPO` tree is intact: `app/core/supernames.py` inserts `v1-tk-version/DefViz-main` into `sys.path` and imports `quantification_analyzer`.
* No `.env` is needed: `app.core.config` has defaults for every setting.
* Importing any `app.domain.services.*` eagerly imports all services – slow, expected. The "Legacy … volumelab not available" messages are harmless (legacy DefQ package is optional).
* **Memory:** labelling + EDT need roughly 8–12 bytes per voxel (int32 labels, float32 distance map; the backend switches to slab labelling / out-of-core EDT automatically for big volumes). Properties are then computed per pore in threads.

In [2]:
import os, sys, json, hashlib, time, math, tempfile, subprocess, warnings
sys.dont_write_bytecode = True
os.environ.setdefault("DEFECT_ATLAS_ACCELERATION_MODE", "cpu")
from pathlib import Path
from datetime import datetime

REPO = Path(REPO); BACKEND = REPO / "backend"
_missing = [p for p in (REPO, BACKEND / "app", REPO / "v1-tk-version" / "DefViz-main") if not p.exists()]
if not _missing:
    pass
else:
    # supernames.py accepts QuantificationAnalyzer-main as an alternative
    alt = REPO / "v1-tk-version" / "QuantificationAnalyzer-main"
    _missing = [p for p in _missing if not (p.name == "DefViz-main" and alt.exists())]
if _missing:
    raise RuntimeError("Defect Atlas tree incomplete, missing: " + ", ".join(map(str, _missing))
                       + "\nThe whole REPO must stay intact (supernames.py needs v1-tk-version).")
sys.path.insert(0, str(BACKEND))

import numpy as np, pandas as pd, psutil
import matplotlib; import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tifffile
import pydantic
if int(pydantic.VERSION.split(".")[0]) >= 2:
    raise RuntimeError(f"pydantic {pydantic.VERSION} found; backend needs 1.x. Use the 'defect-atlas' kernel.")

warnings.filterwarnings("ignore", category=FutureWarning)
from app.domain.models import PipelineConfig
from app.domain.services.pipeline_executor import PipelineExecutor
from app.domain.services.classification import classify_defects, ClassificationConfig
from app.domain.services.shape_classifier import classify_shape
from app.domain.services.projections import compute_projections
from app.services.analytics_service import _volume_key_figures
from app.services.porosity_report_service import compute_direction_report
from app.core.runtime_acceleration import detect_runtime_acceleration
try:
    from app.routers.study import _covariance_ratio_to_axis_ratio as _cov_to_axis_ratio
    _C2A_SRC = "imported from app.routers.study"
except Exception as exc:                                   # router import is optional
    def _cov_to_axis_ratio(value):                         # EXACT copy of study.py:594-597
        if value is None or value <= 0:
            return None
        return math.sqrt(1.0 / value)
    _C2A_SRC = f"replicated from app/routers/study.py:594 (router import failed: {type(exc).__name__})"

def _code_version():
    h = hashlib.md5()
    for p in sorted((BACKEND / "app").rglob("*.py")):
        h.update(p.read_bytes())
    ver = (REPO / "VERSION").read_text().strip() if (REPO / "VERSION").exists() else "?"
    return {"defect_atlas_VERSION": ver, "backend_sources_md5": h.hexdigest()[:12]}
CODE_VERSION = _code_version()


print("python      :", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| pydantic", pydantic.VERSION)
print("backend     :", BACKEND, CODE_VERSION)
print("acceleration:", detect_runtime_acceleration().selected_mode, "(env DEFECT_ATLAS_ACCELERATION_MODE=%s)" % os.environ["DEFECT_ATLAS_ACCELERATION_MODE"])
print("axis ratio  :", _C2A_SRC)
print("RAM         : %.0f GB available of %.0f GB" % (psutil.virtual_memory().available/2**30, psutil.virtual_memory().total/2**30))


/home/alberto.vicente/miniconda3/envs/defect-atlas/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Legacy property modules not available: No module named 'volumelab'
Legacy modules not available: No module named 'volumelab'


python      : 3.11.16 | numpy 1.26.4 | pandas 2.2.3 | pydantic 1.10.17
backend     : /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/10_code/Defect Atlas/backend {'defect_atlas_VERSION': '1.0.0', 'backend_sources_md5': 'f4bb1e073e1b'}
acceleration: cpu (env DEFECT_ATLAS_ACCELERATION_MODE=cpu)
axis ratio  : imported from app.routers.study
RAM         : 224 GB available of 249 GB


### 2b. Helpers (thin wrappers around the backend)

In [3]:
LIST_COLS = ("touching_border_faces", "original_labels", "merge_rules")

def build_config(pores_raw, mask_raw, dims_xyz, res_nm, **over):
    p = dict(connectivity=CONNECTIVITY, min_size_voxels=MIN_SIZE_VOXELS, max_size_voxels=MAX_SIZE_VOXELS,
             enable_merging=ENABLE_MERGING, enable_lof_classification=ENABLE_LOF,
             lof_volume_threshold=LOF_VOLUME_THRESHOLD_VOX, lof_elongation_threshold=LOF_ELONGATION_THRESHOLD,
             lof_distance_threshold=LOF_DISTANCE_THRESHOLD_VOX)
    p.update(over)
    return PipelineConfig(
        pores_volume_path=str(pores_raw), mask_volume_path=str(mask_raw), dimensions=tuple(dims_xyz),
        resolution_nm=float(res_nm), data_type="uint8", mask_data_type="uint8", endian="little", header_offset=0,
        calculate_shortest_distance=True, properties_to_compute=None,
        save_csv_excel=False, save_histograms=False, save_scatter_plots=False, save_pore_rotation_gifs=False,
        save_intermediate_results=False, save_distance_map=False, **p)

def defects_to_table(defects):
    """DefectCollection.to_property_dict_list() = label/type/ranking + all props + merge provenance
    (PipelineExecutor._defects_to_dataframe omits type/ranking/merge info, so it is not used)."""
    df = pd.DataFrame(defects.to_property_dict_list())
    df = df.drop(columns=[c for c in ("coords",) if c in df.columns])       # large array column
    for c in LIST_COLS:
        if c in df.columns:
            df[c] = df[c].map(lambda v: "" if v is None or (isinstance(v, float) and v != v) else
                              (",".join(map(str, v)) if isinstance(v, (list, tuple, set)) else
                               json.dumps(v) if isinstance(v, dict) else str(v)))
    return df

def run_backend(pores_raw, mask_raw, dims_xyz, res_nm, progress=None, **over):
    cfg = build_config(pores_raw, mask_raw, dims_xyz, res_nm, **over)
    errors = cfg.validate()
    if errors:
        raise ValueError("PipelineConfig.validate() errors:\n  - " + "\n  - ".join(errors))
    _volume, defects = PipelineExecutor().run_pipeline(cfg, progress_callback=progress)
    df = defects_to_table(defects)
    del _volume, defects
    return df

BORDER_COLS = ["is_boundary_touching", "is_internal_defect", "touching_border_faces",
               "border_touch_voxel_count", "border_touch_fraction", "border_classification"]

def enrich_table(df, res_nm, lof=None, cpx_threshold=None):
    """Border check + classify_defects + shape class. Returns a new DataFrame."""
    lof = lof or dict(volume=LOF_VOLUME_THRESHOLD_VOX, elongation=LOF_ELONGATION_THRESHOLD, distance=LOF_DISTANCE_THRESHOLD_VOX)
    cpx_threshold = CPX_THRESHOLD if cpx_threshold is None else cpx_threshold
    miss = [c for c in BORDER_COLS if c not in df.columns]
    if miss:   # run_pipeline -> label_volume fills these via _compute_border_props; coords are gone by now
        raise RuntimeError(f"border columns missing from pipeline output: {miss}")
    if "original_labels" in df.columns:
        clusters = set(df.loc[df["original_labels"].astype(str) != "", "label"].astype(int))
    else:
        clusters = set()
    cfg = ClassificationConfig(
        volume_threshold_voxels=lof["volume"] if ENABLE_LOF else float("inf"),
        elongation_threshold=lof["elongation"], distance_threshold_voxels=lof["distance"])
    out = classify_defects(df, cfg, cluster_labels=clusters, resolution_nm=res_nm).df.copy()
    # shape class: Cov_* are covariance ratios in (0, 1]; classify_shape wants axis ratios (>= 1): sqrt(1/ratio)
    out["elong_axis_ratio"] = out["Cov_Elongation"].map(_cov_to_axis_ratio)
    out["flat_axis_ratio"]  = out["Cov_Flatness"].map(_cov_to_axis_ratio)
    e, f, c = out["elong_axis_ratio"].astype(float), out["flat_axis_ratio"].astype(float), out["cpx_factor"].astype(float)
    out["shape_class"] = [classify_shape(a, b, k, cpx_threshold) for a, b, k in zip(e, f, c)]
    out["shape_class_no_cpx"] = [classify_shape(a, b, None) for a, b in zip(e, f)]   # elongation/flatness only
    out["Equivalent Diameter (µm)"] = 2.0 * out["Equivalent Radius (µm)"]
    out["Volume3D (mm³)"] = out["Volume3D (µm³)"] * 1e-9
    return out

def check_units(df, res_nm, rtol=1e-6):
    """Physical-unit sanity checks (asserts). Returns a report DataFrame."""
    res_um = res_nm / 1000.0
    rows = []
    def chk(name, got, expect):
        got, expect = np.asarray(got, float), np.asarray(expect, float)
        ok = np.isfinite(got) & np.isfinite(expect)
        err = float(np.max(np.abs(got[ok] - expect[ok]) / np.maximum(np.abs(expect[ok]), 1e-12))) if ok.any() else np.nan
        rows.append((name, err, bool(err <= rtol) if ok.any() else None))
    chk("Volume3D (µm³) == volume_voxels·(res_nm/1000)³", df["Volume3D (µm³)"], df["volume_voxels"] * res_um**3)
    chk("num_pixels == volume_voxels", df["num_pixels"], df["volume_voxels"])
    chk("Equivalent Radius (µm) == (3V/4π)^(1/3)", df["Equivalent Radius (µm)"], (3 * df["Volume3D (µm³)"] / (4 * math.pi)) ** (1/3))
    for k in ("shortest", "mean", "centroid"):
        v = df[f"{k}_distance_to_surface_vox"]
        chk(f"{k}_distance_to_surface_um == vox·res_um", df[f"{k}_distance_to_surface_um"], v * res_um)
        chk(f"{k}_distance_to_surface (legacy) == vox·res_nm  [legacy unit = nm]", df[f"{k}_distance_to_surface"], v * res_nm)
        if f"{k}_distance_to_surface_nm" in df:
            chk(f"{k}_distance_to_surface_nm == vox·res_nm", df[f"{k}_distance_to_surface_nm"], v * res_nm)
    rep = pd.DataFrame(rows, columns=["check", "max_rel_err", "passed"])
    bad = rep[rep["passed"] == False]
    assert bad.empty, "unit checks failed:\n" + bad.to_string()
    return rep
print("helpers defined")
def convert_to_raw(pores_tif, mask_tif, raw_dir, crop_box, inv_p, inv_m):
    """Stream TIFF pages -> normalised uint8 RAW (0/255) and count voxels on the way."""
    (z0, z1), (y0, y1), (x0, x1) = crop_box
    pr, mr = raw_dir / "pores.raw", raw_dir / "mask.raw"
    cnt = dict(pore_voxels=0, mask_voxels=0, pores_outside_mask=0)
    with tifffile.TiffFile(pores_tif) as tp, tifffile.TiffFile(mask_tif) as tm, open(pr, "wb") as fp, open(mr, "wb") as fm:
        for i in tqdm(range(z0, z1), desc="TIFF -> RAW"):
            a = tp.pages[i].asarray()[y0:y1, x0:x1]; b = tm.pages[i].asarray()[y0:y1, x0:x1]
            P = (a == 0) if inv_p else (a > 0); M = (b == 0) if inv_m else (b > 0)
            cnt["pore_voxels"] += int(P.sum()); cnt["mask_voxels"] += int(M.sum()); cnt["pores_outside_mask"] += int((P & ~M).sum())
            fp.write((P.astype(np.uint8) * 255).tobytes()); fm.write((M.astype(np.uint8) * 255).tobytes())
    return pr, mr, cnt

def _fileid(p): s = Path(p).stat(); return [str(p), s.st_size, int(s.st_mtime)]

def find_jobs(folders):
    """Pair every '*_onlypores.tif' under the folders with its '*_samplemask.tif'. Returns (jobs, problems)."""
    jobs, problems, seen = [], [], set()
    for f in map(Path, folders):
        if not f.is_dir():
            problems.append((str(f), "folder not found")); continue
        pores = sorted(p for p in f.rglob(PORES_GLOB) if OUT_SUBDIR not in p.parts)
        if not pores:
            problems.append((str(f), f"no '{PORES_GLOB}' found")); continue
        for p in pores:
            if p in seen: continue
            seen.add(p)
            m = p.with_name(p.name.replace("_onlypores.tif", MASK_SUFFIX))
            if not m.is_file():
                problems.append((str(p), f"mask not found: {m.name}")); continue
            siblings = [q for q in pores if q.parent == p.parent]
            out = p.parent / (OUT_SUBDIR if len(siblings) == 1 else f"{OUT_SUBDIR}_{p.name[:-len('_onlypores.tif')]}")
            jobs.append(dict(folder=str(f), pores=p, mask=m, out=out, name=p.name[:-len("_onlypores.tif")]))
    return jobs, problems

def voxel_size_for(pores_path):
    for key, v in VOXEL_SIZE_OVERRIDES.items():
        if key in str(pores_path): return float(v)
    return float(VOXEL_SIZE_MM)
print("helpers defined")


helpers defined
helpers defined


## 2c. Per-specimen analysis (function)

The original step-by-step analysis — validate TIFFs, detect polarity, TIFF→RAW, backend pipeline, classification, column/unit checks, specimen summary, plots, export — packaged as `analyse_specimen()` and called once per volume found in `FOLDERS`. Logic is unchanged; only the paths/voxel size/output dir are now arguments. `_display` shows tables inline (and is silent outside Jupyter).

In [4]:
try:
    _display = display
except NameError:
    _display = print

def analyse_specimen(pores_path, mask_path, out_dir, voxel_size_mm):
    PORES_PATH, MASK_PATH = Path(pores_path), Path(mask_path)
    VOXEL_SIZE_MM = voxel_size_mm
    RES_NM = VOXEL_SIZE_MM * 1e6
    VOX_UM = RES_NM / 1000.0
    OUT = Path(out_dir)
    for sub in ("", "_raw", "figures"): (OUT / sub).mkdir(parents=True, exist_ok=True)
    print("voxel size  : %.6g mm = %.6g µm = %.6g nm" % (VOXEL_SIZE_MM, VOX_UM, RES_NM))
    print("OUT_DIR     :", OUT)
    print("free disk at OUT: %.0f GB" % (psutil.disk_usage(str(OUT)).free/2**30))


    def _open(path):
        t = tifffile.TiffFile(path); s = t.series[0]
        if len(s.shape) != 3: raise ValueError(f"{path}: expected a 3-D TIFF stack, got shape {s.shape}")
        if len(t.pages) != s.shape[0]: raise ValueError(f"{path}: {len(t.pages)} pages but series shape {s.shape}; unsupported layout")
        return t, s

    for p in (PORES_PATH, MASK_PATH) + ((Path(GRAY_VOLUME_PATH),) if GRAY_VOLUME_PATH else ()):
        if not p.is_file(): raise FileNotFoundError(f"input volume not found: {p}")

    tp, sp = _open(PORES_PATH); tm, sm = _open(MASK_PATH)
    if sp.shape != sm.shape: raise ValueError(f"pores {sp.shape} and mask {sm.shape} shapes differ")
    if GRAY_VOLUME_PATH:
        tg, sg = _open(Path(GRAY_VOLUME_PATH))
        if sg.shape != sp.shape: raise ValueError(f"gray {sg.shape} differs from pores {sp.shape}")
        tg.close()
    FULL_ZYX = tuple(sp.shape)
    print("pores TIFF:", FULL_ZYX, sp.dtype, "| mask TIFF:", tuple(sm.shape), sm.dtype)

    if CROP is None:
        (z0, z1), (y0, y1), (x0, x1) = [(0, n) for n in FULL_ZYX]
    else:
        (z0, z1), (y0, y1), (x0, x1) = CROP
        assert 0 <= z0 < z1 <= FULL_ZYX[0] and 0 <= y0 < y1 <= FULL_ZYX[1] and 0 <= x0 < x1 <= FULL_ZYX[2], "CROP out of range"

    # ---- front/back wall exclusion. onlypores() only forces the wall slices to "no pore"; the saved sample mask still
    # ---- contains material there, so without this the VVF denominator would include the walls. Keep front <= z < back.
    import re
    WALL_FRONT = WALL_BACK = None
    if WALLS == "auto":
        rep_f = PORES_PATH.parent / PORES_PATH.name.replace("_onlypores.tif", "_report.txt")
        if not rep_f.exists():
            raise FileNotFoundError(f"WALLS='auto' but {rep_f} not found; pass WALLS=(front, back) or WALLS=None")
        txt = rep_f.read_text()
        mf, mb = re.search(r"Front wall slice:\s*(\d+)", txt), re.search(r"Back wall slice:\s*(\d+)", txt)
        if not (mf and mb): raise ValueError(f"could not parse wall slices from {rep_f}")
        WALL_FRONT, WALL_BACK = int(mf.group(1)), int(mb.group(1))
    elif WALLS is not None:
        WALL_FRONT, WALL_BACK = map(int, WALLS)
    if WALL_FRONT is not None:
        lo = max(WALL_FRONT, 0)                                        # onlypores(): wall applied only if > 0
        hi = WALL_BACK if 0 < WALL_BACK < FULL_ZYX[0] else FULL_ZYX[0]
        z0, z1 = max(z0, lo), min(z1, hi)
        assert z1 > z0, "wall exclusion leaves no slices"
        print(f"walls: front={WALL_FRONT}, back={WALL_BACK} -> analysing z in [{z0}, {z1}) of {FULL_ZYX[0]} "
              f"(excluded {z0} front + {FULL_ZYX[0]-z1} back slices from pores AND mask)")
    else:
        print("walls: no wall exclusion")
    Z, Y, X = z1 - z0, y1 - y0, x1 - x0
    DIMS_XYZ = (X, Y, Z)
    N_VOX = X * Y * Z
    print(f"working box (Z,Y,X) = ({Z},{Y},{X})  -> DIMENSIONS (X,Y,Z) = {DIMS_XYZ}; {N_VOX/1e6:.1f} Mvoxels; "
          f"box volume = {N_VOX*VOX_UM**3*1e-9:.3f} mm³")
    need_gb = N_VOX * 10 / 2**30
    print("estimated peak backend memory ≈ %.1f GB (available %.0f GB)" % (need_gb, psutil.virtual_memory().available/2**30))
    if need_gb > psutil.virtual_memory().available / 2**30 * 0.8:
        warnings.warn("estimated backend memory is close to / above available RAM: use CROP or expect swapping")

    # ---- value histograms + polarity on a strided sample of pages
    idx = np.unique(np.linspace(z0, z1 - 1, min(Z, 16)).astype(int))
    cnt_p, cnt_m = {}, {}
    combos = {(ip, im): 0 for ip in (False, True) for im in (False, True)}   # pore voxels outside mask
    ptot = {k: 0 for k in combos}; npx = 0
    for i in idx:
        a = tp.pages[int(i)].asarray()[y0:y1, x0:x1]; b = tm.pages[int(i)].asarray()[y0:y1, x0:x1]
        for arr, c in ((a, cnt_p), (b, cnt_m)):
            u, n = np.unique(arr, return_counts=True)
            for uu, nn in zip(u.tolist(), n.tolist()): c[uu] = c.get(uu, 0) + nn
        npx += a.size
        for (ip, im) in combos:
            P = (a == 0) if ip else (a > 0); M = (b == 0) if im else (b > 0)
            combos[(ip, im)] += int((P & ~M).sum()); ptot[(ip, im)] += int(P.sum())
    print("\nvalue histogram (sampled pages):\n  pores:", dict(sorted(cnt_p.items())[:8]), "\n  mask :", dict(sorted(cnt_m.items())[:8]))
    rows = []
    for (ip, im), out in combos.items():
        tot = ptot[(ip, im)]
        rows.append(dict(invert_pores=ip, invert_mask=im, pore_frac_of_box=tot / npx, pores_outside_mask_frac=(out / tot) if tot else np.nan))
    pol = pd.DataFrame(rows); _display(pol)
    best = pol.sort_values(["pores_outside_mask_frac", "invert_pores", "invert_mask"]).iloc[0]
    INV_P = bool(best.invert_pores) if INVERT_PORES is None else bool(INVERT_PORES)
    INV_M = bool(best.invert_mask)  if INVERT_MASK  is None else bool(INVERT_MASK)
    print(f"POLARITY -> invert_pores={INV_P} ({'auto' if INVERT_PORES is None else 'forced'}), "
          f"invert_mask={INV_M} ({'auto' if INVERT_MASK is None else 'forced'})  "
          f"[pores are {'0' if INV_P else 'non-zero'} in the file, material is {'0' if INV_M else 'non-zero'}]")
    tp.close(); tm.close()

    RAW_DIR = OUT / "_raw"
    raw_key = dict(pores=_fileid(PORES_PATH), mask=_fileid(MASK_PATH), box=[[z0, z1], [y0, y1], [x0, x1]], inv=[INV_P, INV_M])
    meta_f = RAW_DIR / "raw_meta.json"
    PORES_RAW, MASK_RAW = RAW_DIR / "pores.raw", RAW_DIR / "mask.raw"
    cached = None
    if USE_CACHE and meta_f.exists() and PORES_RAW.exists() and MASK_RAW.exists():
        m = json.loads(meta_f.read_text())
        if m["key"] == raw_key and PORES_RAW.stat().st_size == N_VOX == MASK_RAW.stat().st_size: cached = m["counts"]
    if cached is None:
        t0 = time.time(); PORES_RAW, MASK_RAW, COUNTS = convert_to_raw(PORES_PATH, MASK_PATH, RAW_DIR, ((z0, z1), (y0, y1), (x0, x1)), INV_P, INV_M)
        meta_f.write_text(json.dumps(dict(key=raw_key, counts=COUNTS))); print("converted in %.0fs" % (time.time() - t0))
    else:
        COUNTS = cached; print("RAW cache reused")

    # expected bytes = X*Y*Z*itemsize + header(0)
    for p in (PORES_RAW, MASK_RAW):
        assert p.stat().st_size == N_VOX, f"{p}: {p.stat().st_size} bytes, expected {N_VOX}"
    MASK_VOX, PORE_VOX_TOTAL = COUNTS["mask_voxels"], COUNTS["pore_voxels"]
    print(f"\nRAW sizes OK ({N_VOX} bytes each)")
    print(f"pore voxels     : {PORE_VOX_TOTAL:,}  ({100*PORE_VOX_TOTAL/N_VOX:.3f}% of box)")
    print(f"material voxels : {MASK_VOX:,}  ({100*MASK_VOX/N_VOX:.2f}% of box)")
    print(f"pore voxels outside mask: {COUNTS['pores_outside_mask']:,}")
    print(f"VVF (pore voxels / material voxels) = {100*PORE_VOX_TOTAL/MASK_VOX:.4f} %")
    rep = PORES_PATH.parent / (PORES_PATH.name.replace("_onlypores.tif", "_report.txt"))
    if rep.exists() and CROP is None and WALL_FRONT is None:
        import re
        m_ = re.search(r"Void volumetric fraction.*?\(([\d.]+)%\)", rep.read_text())
        print(f"onlypores report.txt says VVF = {m_.group(1)} %  (cross-check)" if m_ else "report.txt found, VVF line not parsed")

    # mid-slice preview from the normalised RAW files
    pm = np.memmap(PORES_RAW, np.uint8, "r", shape=(Z, Y, X)); mm = np.memmap(MASK_RAW, np.uint8, "r", shape=(Z, Y, X))
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    k = Z // 2
    ax[0].imshow(pm[k], cmap="gray"); ax[0].set_title(f"pores (z={k})")
    ax[1].imshow(mm[k], cmap="gray"); ax[1].set_title(f"material mask (z={k})")
    ov = np.zeros((Y, X, 3), np.float32); ov[..., 1] = mm[k] > 0; ov[..., 0] = pm[k] > 0; ov[..., 2] = 0
    ax[2].imshow(ov); ax[2].set_title("overlay: red = pores, green = material")
    for a_ in ax: a_.set_xlabel("x (voxel)"); a_.set_ylabel("y (voxel)")
    plt.tight_layout(); (plt.show() if SHOW_PLOTS else plt.close('all'))

    cfg = build_config(PORES_RAW, MASK_RAW, DIMS_XYZ, RES_NM)
    errs = cfg.validate()
    print("PipelineConfig.validate() ->", errs)
    assert errs == [], "fix the errors above before running"

    run_key = dict(raw=raw_key, res_nm=RES_NM, min=MIN_SIZE_VOXELS, max=MAX_SIZE_VOXELS, conn=CONNECTIVITY,
                   merge=ENABLE_MERGING, code=CODE_VERSION)
    RAW_TBL, RAW_KEY = OUT / "pores_raw.parquet", OUT / "pores_raw.key.json"
    if USE_CACHE and RAW_TBL.exists() and RAW_KEY.exists() and json.loads(RAW_KEY.read_text()) == run_key:
        df_raw = pd.read_parquet(RAW_TBL); RUNTIME_S = None
        print(f"pipeline result loaded from cache: {RAW_TBL}  ({len(df_raw):,} pores)")
    else:
        bar = tqdm(total=100, desc="pipeline", bar_format="{l_bar}{bar}| {n:.0f}% {postfix}")
        def progress(msg, pct):
            bar.n = min(100, float(pct)); bar.set_postfix_str(str(msg)[:60]); bar.refresh()
        t0 = time.time()
        df_raw = run_backend(PORES_RAW, MASK_RAW, DIMS_XYZ, RES_NM, progress=progress)
        RUNTIME_S = time.time() - t0; bar.n = 100; bar.refresh(); bar.close()
        df_raw.to_parquet(RAW_TBL); RAW_KEY.write_text(json.dumps(run_key))
        print(f"{len(df_raw):,} pores in {RUNTIME_S:.0f}s; cached to {RAW_TBL}")

    df = enrich_table(df_raw, RES_NM)
    print("type        :", df["type"].value_counts().to_dict())
    print("shape_class :", df["shape_class"].value_counts(dropna=False).to_dict())
    print("no_cpx class:", df["shape_class_no_cpx"].value_counts(dropna=False).to_dict())
    print("border      :", df["border_classification"].value_counts().to_dict())

    pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
    _display(df.head())
    _display(df.describe().T)
    print(f"{len(df):,} pores x {df.shape[1]} columns")

    EXPECTED = {
     "Size": ["Volume3D (µm³)", "volume_voxels", "num_pixels", "Equivalent Radius (µm)", "Equivalent Diameter (µm)"],
     "Surface": ["Area3D (µm²)", "border_voxel_count", "Area3D_capped", "Sphericity"],
     "Position": ["centroid-0", "centroid-1", "centroid-2"] + [f"bbox-{i}" for i in range(6)] + ["BoundingBox_Dx (µm)", "BoundingBox_Dy (µm)", "BoundingBox_Dz (µm)"],
     "Feret": ["Length3D (µm)", "Width3D (µm)", "Breadth3D (µm)", "Thickness3D (µm)"],
     "Ellipsoid": ["Ellipsoid_Major_Axis_Length (µm)", "Ellipsoid_Medium_Axis_Length (µm)", "Ellipsoid_Minor_Axis_Length (µm)",
                   "Ellipsoid_Semi_Axis_1 (µm)", "Ellipsoid_Semi_Axis_2 (µm)", "Ellipsoid_Semi_Axis_3 (µm)"]
                   + [f"OrientedBBox_Corner{i}_{a} (µm)" for i in range(1, 9) for a in "xyz"],
     "Tensor shape": [f"{p}_{q}" for p in ("Cov", "Inertia") for q in ("Elongation", "Flatness", "Anisotropy", "elongation_avg", "flatness_avg")]
                   + [f"Covariance_Eigenvalue_{i}" for i in (1, 2, 3)] + [f"Inertia_Eigenvalue_{i}" for i in (1, 2, 3)]
                   + [f"Covariance_Eigenvector_{i}_{a}" for i in (1, 2, 3) for a in "xyz"] + [f"Inertia_Eigenvector_{i}_{a}" for i in (1, 2, 3) for a in "xyz"],
     "Orientation": ["Phi (degrees)", "Theta (degrees)", "Cov_Phi (degrees)", "Cov_Theta (degrees)", "Inertia_Phi (degrees)", "Inertia_Theta (degrees)"],
     "Complexity": ["cpx_factor", "cpx_factor_capped"],
     "Distance to surface": [f"{k}_distance_to_surface{s}" for k in ("shortest", "mean", "centroid") for s in ("_vox", "", "_nm", "_um")],
     "Border": BORDER_COLS,
     "Classification": ["type", "ranking", "shape_class", "shape_class_no_cpx"],
    }
    if ENABLE_MERGING: EXPECTED["Merge provenance"] = ["original_labels", "merge_rules"]
    rows = []
    for grp, cols in EXPECTED.items():
        for c in cols:
            if c not in df.columns: rows.append((grp, c, "MISSING", np.nan))
            else:
                n_nan = int(df[c].isna().sum())
                rows.append((grp, c, "ALL NaN" if n_nan == len(df) and len(df) else "ok", n_nan))
    chk = pd.DataFrame(rows, columns=["group", "column", "status", "n_nan"])
    print(chk.groupby(["group", "status"]).size().unstack(fill_value=0).to_string())
    _display(chk[chk.status != "ok"])
    extra = [c for c in df.columns if c not in set(sum(EXPECTED.values(), []))]
    print("columns present but not in the spec groups:", extra)

    # missing-values report (columns with any NaN)
    nn = df.isna().sum(); nn = nn[nn > 0].sort_values(ascending=False)
    print(f"\ncolumns with NaNs: {len(nn)}"); _display((pd.DataFrame({"n_nan": nn, "pct": 100 * nn / len(df)})).head(30))

    # capped / skipped-for-large-pore flags stay visible
    for c in ("Area3D_capped", "cpx_factor_capped"):
        if c in df: print(f"{c}: {int(df[c].astype(bool).sum())} of {len(df)} pores capped")

    units_report = check_units(df, RES_NM)
    _display(units_report)
    print("ALL UNIT CHECKS PASSED  (volume in µm³, legacy distance columns in nm, *_um in µm, *_vox in voxels)")

    def specimen_summary(df, mask_vox, n_vox, pore_vox_total, res_nm):
        vox_um3 = (res_nm / 1000.0) ** 3
        kf = _volume_key_figures(df)                                  # analytics_service: d = (6V/π)^(1/3)
        pore_um3 = float(df["Volume3D (µm³)"].sum())
        mat_mm3, box_mm3 = mask_vox * vox_um3 * 1e-9, n_vox * vox_um3 * 1e-9
        internal = df[df["is_internal_defect"].astype(bool)]
        s = dict(
            voxel_size_um=res_nm / 1000.0, box_voxels=int(n_vox), box_volume_mm3=box_mm3,
            material_voxels=int(mask_vox), material_volume_mm3=mat_mm3,
            defect_count=int(len(df)),
            porosity_pct=100.0 * pore_um3 * 1e-9 / mat_mm3,                 # study.py / routers/pipeline.py: pore volume / mask volume
            porosity_box_pct=100.0 * pore_um3 * 1e-9 / box_mm3,
            porosity_internal_only_pct=100.0 * float(internal["Volume3D (µm³)"].sum()) * 1e-9 / mat_mm3,
            defect_density_per_mm3=len(df) / mat_mm3,
            total_defect_volume_um3=pore_um3, total_defect_volume_mm3=pore_um3 * 1e-9,
            largest_volume_um3=kf.get("largest_volume_um3"), largest_eq_diameter_um=kf.get("largest_eq_diameter_um"),
            median_eq_diameter_um=kf.get("median_eq_diameter_um"),
            pore_voxels_in_table=int(df["volume_voxels"].sum()), pore_voxels_in_volume=int(pore_vox_total),
            counts_by_type=df["type"].value_counts().to_dict(),
            counts_by_shape_class=df["shape_class"].value_counts(dropna=False).rename(lambda k: str(k)).to_dict(),
            counts_by_shape_class_no_cpx=df["shape_class_no_cpx"].value_counts(dropna=False).rename(lambda k: str(k)).to_dict(),
            counts_by_border_class=df["border_classification"].value_counts().to_dict(),
            n_boundary_touching=int(df["is_boundary_touching"].astype(bool).sum()),
            n_area_capped=int(df["Area3D_capped"].astype(bool).sum()) if "Area3D_capped" in df else None,
            n_cpx_capped=int(df["cpx_factor_capped"].astype(bool).sum()) if "cpx_factor_capped" in df else None,
        )
        return s

    SUMMARY = specimen_summary(df, MASK_VOX, N_VOX, PORE_VOX_TOTAL, RES_NM)
    _display(pd.Series({k: v for k, v in SUMMARY.items() if not isinstance(v, dict)}, name="value").to_frame())
    for k in ("counts_by_type", "counts_by_shape_class", "counts_by_shape_class_no_cpx", "counts_by_border_class"):
        print(k, SUMMARY[k])
    d_vox = SUMMARY["pore_voxels_in_volume"] - SUMMARY["pore_voxels_in_table"]
    print(f"\npore voxels in volume − in table = {d_vox:,}  (non-zero only if MIN_SIZE/MAX_SIZE filtering removed pores)")

    FIG = OUT / "figures"
    def save(fig, name):
        fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight")
        if SHOW_PLOTS: plt.show()
        else: plt.close(fig)
    TYPE_COL = {"Pore": "#1f77b4", "LoF": "#d62728", "Cluster": "#2ca02c", "Border": "#ff7f0e"}

    # data-quality flags (shown, not hidden): covariance ratios should lie in (0, 1]; sphericity should be <= 1
    n_cov_bad = int(((df["Cov_Elongation"] <= 0) | (df["Cov_Flatness"] <= 0)).sum()); n_cov_nan = int(df["Cov_Elongation"].isna().sum())
    n_sph_gt1 = int((df["Sphericity"] > 1.0 + 1e-9).sum()); n_1vox = int((df["volume_voxels"] <= 1).sum())
    print(f"Cov_Elongation/Cov_Flatness <= 0 (numerically degenerate, no shape class): {n_cov_bad}; NaN: {n_cov_nan}")
    print(f"Sphericity > 1 (voxelised area of tiny pores): {n_sph_gt1}; pores with <= 1 voxel: {n_1vox}  (use MIN_SIZE_VOXELS to drop them)")
    SUMMARY.update(n_cov_ratio_nonpositive=n_cov_bad, n_cov_ratio_nan=n_cov_nan, n_sphericity_gt1=n_sph_gt1, n_single_voxel_pores=n_1vox)
    vol = df["Volume3D (µm³)"]; vol = vol[vol > 0]
    fig, ax = plt.subplots(2, 2, figsize=(12, 8))
    ax[0, 0].hist(vol, bins=np.logspace(np.log10(vol.min()), np.log10(vol.max()), 60)); ax[0, 0].set_xscale("log"); ax[0, 0].set_yscale("log")
    ax[0, 0].set_xlabel("Volume (µm³)"); ax[0, 0].set_ylabel("pore count"); ax[0, 0].set_title("Volume")
    ed = df["Equivalent Diameter (µm)"]; ax[0, 1].hist(ed.dropna(), bins=60); ax[0, 1].set_yscale("log")
    ax[0, 1].set_xlabel("Equivalent diameter (µm)"); ax[0, 1].set_ylabel("pore count"); ax[0, 1].set_title("Equivalent diameter")
    ax[1, 0].hist(df["Sphericity"].dropna(), bins=60); ax[1, 0].set_xlabel("Sphericity (-)"); ax[1, 0].set_ylabel("pore count"); ax[1, 0].set_title("Sphericity")
    ax[1, 1].hist(df["Cov_Elongation"].dropna(), bins=60, range=(0, 1), alpha=.7, label="Cov_Elongation (λ2/λ1)")
    ax[1, 1].hist(df["Cov_Flatness"].dropna(), bins=60, range=(0, 1), alpha=.7, label="Cov_Flatness (λ3/λ2)")
    ax[1, 1].set_xlabel("covariance ratio (-)"); ax[1, 1].set_ylabel("pore count"); ax[1, 1].legend(); ax[1, 1].set_title("Elongation / flatness")
    plt.tight_layout(); save(fig, "hist_volume_diameter_sphericity_elongation")

    fig, ax = plt.subplots(figsize=(8, 6))
    for t, g in df.groupby("type"):
        ax.scatter(g["Volume3D (µm³)"], g["Sphericity"], s=6, alpha=.4, c=TYPE_COL.get(t, "grey"), label=f"{t} ({len(g):,})", rasterized=True)
    ax.set_xscale("log"); ax.set_xlabel("Volume (µm³)"); ax.set_ylabel("Sphericity (-)"); ax.legend(markerscale=3); ax.set_title("Volume vs sphericity by type")
    save(fig, "scatter_volume_sphericity_by_type")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for a_, col, rng in ((ax[0], "Phi", (-180, 180)), (ax[1], "Theta", (0, 180))):
        for pre, lab in (("", "main"), ("Cov_", "cov"), ("Inertia_", "inertia")):
            v = df[f"{pre}{col} (degrees)"].dropna()
            a_.hist(v, bins=60, histtype="step", label=f"{lab} (n={len(v):,})")
        a_.set_xlabel(f"{col} (degrees)"); a_.set_ylabel("pore count"); a_.legend(); a_.set_title(f"Orientation: {col}")
    plt.tight_layout(); save(fig, "hist_phi_theta")

    # per-slice porosity (backend: porosity_report_service.compute_direction_report), RAW files are already 0/255 => invert=False
    SLICE = {}
    if RUN_SLICE_PROFILE:
        for axis in SLICE_AXES:
            f = OUT / f"slice_profile_{axis}.csv"; kf_ = OUT / f"slice_profile_{axis}.key.json"
            if USE_CACHE and f.exists() and kf_.exists() and json.loads(kf_.read_text()) == raw_key:
                SLICE[axis] = pd.read_csv(f); continue
            bar = tqdm(total=1, desc=f"slices {axis}")
            def cb(done, total): bar.total = total; bar.n = done; bar.refresh()
            SLICE[axis] = pd.DataFrame(compute_direction_report(pm, mm, axis, VOX_UM, False, False, cb)); bar.close()
            SLICE[axis].to_csv(f, index=False); kf_.write_text(json.dumps(raw_key))
        fig, ax = plt.subplots(1, len(SLICE), figsize=(5.5 * len(SLICE), 4), squeeze=False)
        for a_, (axis, d) in zip(ax[0], SLICE.items()):
            a_.plot(d["slice_um"], d["porosity_percent"]); a_.set_xlabel(f"{axis} position (µm)"); a_.set_ylabel("porosity, area % of material (%)")
            a_.set_title(f"Porosity profile along {axis.upper()}")
        plt.tight_layout(); save(fig, "porosity_profile_slices")
        print({a: dict(mean_pct=float(d["porosity_percent"].mean()), max_pct=float(d["porosity_percent"].max())) for a, d in SLICE.items()})
    else:
        print("RUN_SLICE_PROFILE=False")

    # porosity vs Z from the pore table (centroid slab volume / material volume per slab), independent cross-check
    zc = df["centroid-0"].to_numpy(float); zv = df["Volume3D (µm³)"].to_numpy(float)
    nb = min(40, Z); edges = np.linspace(0, Z, nb + 1)
    mat_z = np.array([np.count_nonzero(mm[int(a):int(b)]) for a, b in zip(edges[:-1], edges[1:])], float) * VOX_UM**3
    slab_por = 100 * np.histogram(zc, edges, weights=zv)[0] / np.maximum(mat_z, 1)
    fig, ax = plt.subplots(figsize=(7, 4)); ax.step(edges[:-1] * VOX_UM, slab_por, where="post")
    ax.set_xlabel("z position (µm)"); ax.set_ylabel("slab porosity (%)"); ax.set_title("Porosity vs Z (pore centroid slabs)")
    save(fig, "porosity_vs_z_slabs")

    if RUN_PROJECTION:
        pr = compute_projections(pm)           # streamed along Z in slabs
        fig, ax = plt.subplots(1, 2, figsize=(13, 6))
        im0 = ax[0].imshow(pr.max_pores, cmap="gray_r"); ax[0].set_title("Z max projection (any pore in column)")
        im1 = ax[1].imshow(pr.avg_pores, cmap="viridis"); ax[1].set_title("Z average projection"); plt.colorbar(im1, ax=ax[1], label="% of Z height that is pore")
        for a_ in ax: a_.set_xlabel("x (voxel)"); a_.set_ylabel("y (voxel)")
        plt.tight_layout(); save(fig, "projection_z")

    cols_out = [c for c in df.columns if c != "coords"]
    df[cols_out].to_csv(OUT / "pores.csv", index=False)
    df[cols_out].to_parquet(OUT / "pores.parquet", index=False)
    (OUT / "specimen_summary.json").write_text(json.dumps(SUMMARY, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
    run_config = dict(
        created=datetime.now().isoformat(timespec="seconds"), code_version=CODE_VERSION, repo=str(REPO),
        kernel_python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__, pydantic=pydantic.VERSION,
        inputs=dict(walls=[WALL_FRONT, WALL_BACK], analysed_z_range=[z0, z1], pores=str(PORES_PATH), mask=str(MASK_PATH), gray=GRAY_VOLUME_PATH, crop=CROP, full_shape_zyx=list(FULL_ZYX),
                    working_shape_zyx=[Z, Y, X]),
        polarity=dict(invert_pores=INV_P, invert_mask=INV_M, auto_pores=INVERT_PORES is None, auto_mask=INVERT_MASK is None),
        voxel_size_mm=VOXEL_SIZE_MM, resolution_nm=RES_NM,
        pipeline=dict(connectivity=CONNECTIVITY, min_size_voxels=MIN_SIZE_VOXELS, max_size_voxels=MAX_SIZE_VOXELS,
                      enable_merging=ENABLE_MERGING, enable_lof=ENABLE_LOF, calculate_shortest_distance=True, properties_to_compute=None),
        classification=dict(volume_threshold_voxels=LOF_VOLUME_THRESHOLD_VOX, elongation_threshold=LOF_ELONGATION_THRESHOLD,
                            distance_threshold_voxels=LOF_DISTANCE_THRESHOLD_VOX, cpx_threshold=CPX_THRESHOLD,
                            axis_ratio_conversion="sqrt(1/cov_ratio), " + _C2A_SRC),
        acceleration=os.environ["DEFECT_ATLAS_ACCELERATION_MODE"], pipeline_runtime_s=RUNTIME_S,
        slice_axes=list(SLICE_AXES) if RUN_SLICE_PROFILE else None,
    )
    (OUT / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str))
    if not KEEP_RAW:
        for p in (PORES_RAW, MASK_RAW): Path(p).unlink(missing_ok=True)
        (RAW_DIR / "raw_meta.json").unlink(missing_ok=True)
    print("written to", OUT); [print("  ", p.relative_to(OUT)) for p in sorted(OUT.rglob("*")) if p.is_file() and "_raw" not in p.parts]

    del pm, mm
    import gc; gc.collect()
    flat = {k: v for k, v in SUMMARY.items() if not isinstance(v, dict)}
    for k in ("counts_by_type", "counts_by_shape_class", "counts_by_border_class"):
        flat[k] = json.dumps(SUMMARY[k], default=str)
    return flat


## 3. Discover volumes in the folders

In [5]:
JOBS, PROBLEMS = find_jobs(FOLDERS)
for j in JOBS: print(f"{j['name']:<30} pores={j['pores'].name}  mask={j['mask'].name}  voxel={voxel_size_for(j['pores'])} mm  -> {j['out']}")
for p, why in PROBLEMS: print("SKIPPED:", p, "-", why)
print(f"\n{len(JOBS)} volume(s) to analyse, {len(PROBLEMS)} problem(s)")


volume_eq_rotated_aligned      pores=volume_eq_rotated_aligned_onlypores.tif  mask=volume_eq_rotated_aligned_samplemask.tif  voxel=0.025 mm  -> /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/4/onlypores files/pore_analysis
volume_eq_rotated_aligned      pores=volume_eq_rotated_aligned_onlypores.tif  mask=volume_eq_rotated_aligned_samplemask.tif  voxel=0.025 mm  -> /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/5/onlypores files/pore_analysis
volume_eq_aligned              pores=volume_eq_aligned_onlypores.tif  mask=volume_eq_aligned_samplemask.tif  voxel=0.025 mm  -> /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/7/onlypores files/pore_analysis
volume_eq_aligned              pores=volume_eq_aligned_onlypores.tif  mask=volume_eq_aligned_samplemask.tif  voxel=0.025 mm  -> /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/8/on

## 4. Run the analysis on every folder

In [6]:
RESULTS = [dict(folder=p, name=None, status="skipped: " + why) for p, why in PROBLEMS]
for n, job in enumerate(JOBS, 1):
    print("\n" + "=" * 100 + f"\n[{n}/{len(JOBS)}] {job['name']}  ({job['folder']})\n" + "=" * 100)
    t0 = time.time()
    try:
        flat = analyse_specimen(job["pores"], job["mask"], job["out"], voxel_size_for(job["pores"]))
        RESULTS.append(dict(folder=job["folder"], name=job["name"], out_dir=str(job["out"]), status="ok",
                            elapsed_s=round(time.time() - t0, 1), **flat))
    except Exception as exc:
        import traceback; traceback.print_exc()
        RESULTS.append(dict(folder=job["folder"], name=job["name"], out_dir=str(job["out"]),
                            status=f"error: {type(exc).__name__}: {exc}", elapsed_s=round(time.time() - t0, 1)))
        if STOP_ON_ERROR: raise
    plt.close("all")



[1/7] volume_eq_rotated_aligned  (/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/4/onlypores files)
voxel size  : 0.025 mm = 25 µm = 25000 nm
OUT_DIR     : /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/4/onlypores files/pore_analysis
free disk at OUT: 5368 GB
pores TIFF: (203, 3226, 1609) uint8 | mask TIFF: (203, 3226, 1609) uint8
walls: front=9, back=174 -> analysing z in [9, 174) of 203 (excluded 9 front + 29 back slices from pores AND mask)
working box (Z,Y,X) = (165,3226,1609)  -> DIMENSIONS (X,Y,Z) = (1609, 3226, 165); 856.5 Mvoxels; box volume = 13382.103 mm³
estimated peak backend memory ≈ 8.0 GB (available 225 GB)

value histogram (sampled pages):
  pores: {0: 82823928, 255: 226216} 
  mask : {0: 7666756, 255: 75383388}


,invert_pores,invert_mask,pore_frac_of_box,pores_outside_mask_frac
0,False,False,0.002724,0.000000
1,False,True,0.002724,1.000000
2,True,False,0.997276,0.092567
3,True,True,0.997276,0.907433


POLARITY -> invert_pores=False (auto), invert_mask=False (auto)  [pores are non-zero in the file, material is non-zero]


TIFF -> RAW: 100%|██████████| 165/165 [00:20<00:00,  7.88it/s]
/tmp/ipykernel_2127584/872339919.py:141: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); (plt.show() if SHOW_PLOTS else plt.close('all'))


converted in 21s

RAW sizes OK (856454610 bytes each)
pore voxels     : 5,811,857  (0.679% of box)
material voxels : 778,617,527  (90.91% of box)
pore voxels outside mask: 0
VVF (pore voxels / material voxels) = 0.7464 %
PipelineConfig.validate() -> []


...

ALL UNIT CHECKS PASSED  (volume in µm³, legacy distance columns in nm, *_um in µm, *_vox in voxels)


,value
voxel_size_um,2.500000e+01
box_voxels,8.482008e+08
box_volume_mm3,1.325314e+04
material_voxels,8.180656e+08
material_volume_mm3,1.278228e+04
defect_count,7.304000e+04
porosity_pct,7.115509e-01
porosity_box_pct,6.862706e-01
porosity_internal_only_pct,6.334829e-01
defect_density_per_mm3,5.714163e+00


counts_by_type {'Pore': 72047, 'LoF': 950, 'Border': 43}
counts_by_shape_class {'None': 50808, 'Complex': 22189, 'Ellipsoidal': 32, 'Disc': 9, 'Spherical': 2}
counts_by_shape_class_no_cpx {'Ellipsoidal': 48564, 'Disc': 13652, 'Rod': 10786, 'Spherical': 33, 'None': 5}
counts_by_border_class {'internal': 72974, 'single_face': 64, 'edge': 2}

pore voxels in volume − in table = 0  (non-zero only if MIN_SIZE/MAX_SIZE filtering removed pores)
Cov_Elongation/Cov_Flatness <= 0 (numerically degenerate, no shape class): 5; NaN: 0
Sphericity > 1 (voxelised area of tiny pores): 7205; pores with <= 1 voxel: 0  (use MIN_SIZE_VOXELS to drop them)


/tmp/ipykernel_2127584/872339919.py:256: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  if SHOW_PLOTS: plt.show()
/tmp/ipykernel_2127584/872339919.py:256: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  if SHOW_PLOTS: plt.show()
slices x: 100%|██████████| 1593/1593 [00:10<00:00, 147.29it/s]


{'z': {'mean_pct': 0.7107273688154156, 'max_pct': 3.8811220043310004}, 'y': {'mean_pct': 0.7112214219812912, 'max_pct': 2.1819409393873275}, 'x': {'mean_pct': 1.0307172809255258, 'max_pct': 99.67058823529412}}
written to /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/12/onlypores files/pore_analysis
   figures/hist_phi_theta.png
   figures/hist_volume_diameter_sphericity_elongation.png
   figures/porosity_profile_slices.png
   figures/porosity_vs_z_slabs.png
   figures/projection_z.png
   figures/scatter_volume_sphericity_by_type.png
   pores.csv
   pores.parquet
   pores_raw.key.json
   pores_raw.parquet
   run_config.json
   slice_profile_x.csv
   slice_profile_x.key.json
   slice_profile_y.csv
   slice_profile_y.key.json
   slice_profile_z.csv
   slice_profile_z.key.json
   specimen_summary.json


## 5. Batch summary

In [7]:
BATCH = pd.DataFrame(RESULTS)
if BATCH_SUMMARY_PATH: _bp = Path(BATCH_SUMMARY_PATH)
else: _bp = Path(JOBS[0]["out"]).parent / "pore_analysis_batch_summary.csv" if JOBS else Path(FOLDERS[0]) / "pore_analysis_batch_summary.csv"
if len(BATCH):
    BATCH.to_csv(_bp, index=False); print("batch summary written to", _bp)
    _cols = [c for c in ("name", "status", "elapsed_s", "voxel_size_um", "defect_count", "porosity_pct", "porosity_internal_only_pct",
                         "defect_density_per_mm3", "largest_eq_diameter_um", "median_eq_diameter_um") if c in BATCH.columns]
    _display(BATCH[_cols])
n_bad = int((~BATCH["status"].eq("ok")).sum()) if len(BATCH) else 0
print(f"{len(BATCH) - n_bad} ok, {n_bad} failed/skipped")


batch summary written to /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Juan Ignacio/probetas/4/onlypores files/pore_analysis_batch_summary.csv


,name,status,elapsed_s,voxel_size_um,defect_count,porosity_pct,porosity_internal_only_pct,defect_density_per_mm3,largest_eq_diameter_um,median_eq_diameter_um
0,volume_eq_rotated_aligned,ok,92.3,25.0,2770,0.746433,0.677690,0.227686,1590.657852,196.692226
1,volume_eq_rotated_aligned,ok,90.6,25.0,4723,0.895458,0.894053,0.380227,1826.063127,155.910384
2,volume_eq_aligned,ok,85.3,25.0,2023,0.533916,0.514368,0.158091,1924.433114,140.024193
3,volume_eq_aligned,ok,92.6,25.0,4930,1.074017,1.038115,0.389471,2057.869251,169.780738
4,volume_eq_aligned,ok,415.3,25.0,145691,0.819737,0.739496,11.560417,2109.325819,71.012404
5,volume_eq_aligned,ok,427.6,25.0,149488,0.865254,0.796037,11.593234,2055.468223,71.012404
6,volume_eq_aligned,ok,244.8,25.0,73040,0.711551,0.633483,5.714163,1818.138472,71.012404


7 ok, 0 failed/skipped


## 6. How to reuse

1. Edit only the **Parameters** cell: list the specimen folders in `FOLDERS` and set `VOXEL_SIZE_MM` (default) / `VOXEL_SIZE_OVERRIDES` (per specimen). Use the `defect-atlas` kernel.
2. *Run All*. Each folder is searched recursively for `*_onlypores.tif` + the matching `*_samplemask.tif`; folders with no pair, or a missing mask, are reported and skipped. A failure in one specimen is logged and the loop continues (`STOP_ON_ERROR=True` to abort instead).
3. Outputs per specimen in `<pores folder>/pore_analysis/` (or `pore_analysis_<prefix>/` if a folder holds several volumes): `pores.csv/.parquet`, `specimen_summary.json`, `slice_profile_*.csv`, `figures/*.png`, `run_config.json`, `_raw/`. Across specimens: `pore_analysis_batch_summary.csv`.
4. Re-runs are cheap (per-specimen cache, same invalidation rules as before; `USE_CACHE=False` forces a rerun). `SHOW_PLOTS=False` saves figures without rendering them inline.
5. Units: volumes µm³, lengths µm, `*_distance_to_surface` (no suffix) in **nm**, `*_vox` in voxels, `centroid-*`/`bbox-*` in voxels `(z, y, x)`.
6. Caveats unchanged: `shape_class` is dominated by the `cpx_factor` override — prefer `shape_class_no_cpx`; tiny pores can give NaN covariance ratios / Sphericity > 1 (`MIN_SIZE_VOXELS`). The notebook never writes inside the Defect Atlas repo.
